<div class="alert alert-block alert-info" style="border-left: 7px solid #20639B; border-radius: 4px; padding: 20px;">
    <h2 style="color: #20639B; margin-top: 0; font-weight: bold;">
        🚀 Welcome to the Deep Learning Sequence Guide!
    </h2>
    <p style="font-size: 19px; line-height: 1.6; color: #333;">
        This Kaggle notebook provides a <b>hands-on guide</b> to building, understanding, and training 
        <span style="color:#ED553B; font-weight:bold;">Vanilla Elman RNNs</span> and 
        <span style="color:#3CAEA3; font-weight:bold;">LSTMs</span> from scratch using <b>PyTorch</b>. 
    </p>
    <p style="font-size: 15px; line-height: 1.6; color: #333;">
        It features complete, practical implementations including <b>forward passes</b>, recurrent mechanics, 
        and fully functional <b>training workflows for text generation tasks</b>.
    </p>
    <hr style="border: 0; border-top: 1px solid #d9edf7; margin: 15px 0;">
    <p style="font-size: 15px; margin-bottom: 0; font-weight: bold;">
        👉 Explore the interactive code blocks and run the models directly on 
        <a href="https://www.kaggle.com/code/saibhossain/building-vanilla-elman-rnn-from-scratch" target="_blank" style="color: #20639B; text-decoration: underline;">
            Kaggle Notebook 🔗
        </a>
    </p>
</div>


# RNN

In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


In [2]:
text = """
Albert Einstein was born at Ulm, in Württemberg, Germany, on March 14, 1879.\n
Six weeks later the family moved to Munich, where he later on began his schooling at the Luitpold Gymnasium.\n
Later, they moved to Italy and Albert continued his education at Aarau, Switzerland and in 1896 he entered the Swiss Federal Polytechnic School in Zurich to be trained as a teacher in physics and mathematics.\n
In 1901, the year he gained his diploma, he acquired Swiss citizenship and, as he was unable to find a teaching post, he accepted a position as technical assistant in the Swiss Patent Office.\n
In 1905 he obtained his doctor’s degree.\n
During his stay at the Patent Office, and in his spare time, he produced much of his remarkable work and in 1908 he was appointed Privatdozent in Berne.\n
In 1909 he became Professor Extraordinary at Zurich, in 1911 Professor of Theoretical Physics at Prague, returning to Zurich in the following year to fill a similar post.\n
In 1914 he was appointed Director of the Kaiser Wilhelm Physical Institute and Professor in the University of Berlin. He became a German citizen in 1914 and remained in Berlin until 1933 when he renounced his citizenship for political reasons and emigrated to America to take the position of Professor of Theoretical Physics at Princeton*. He became a United States citizen in 1940 and retired from his post in 1945.\n
After World War II, Einstein was a leading figure in the World Government Movement, he was offered the Presidency of the State of Israel, which he declined, and he collaborated with Dr. Chaim Weizmann in establishing the Hebrew University of Jerusalem.\n
Einstein always appeared to have a clear view of the problems of physics and the determination to solve them.\n
He had a strategy of his own and was able to visualize the main stages on the way to his goal.\n
He regarded his major achievements as mere stepping-stones for the next advance.\n
At the start of his scientific work, Einstein realized the inadequacies of Newtonian mechanics and his special theory of relativity stemmed from an attempt to reconcile the laws of mechanics with the laws of the electromagnetic field.\n
He dealt with classical problems of statistical mechanics and problems in which they were merged with quantum theory: this led to an explanation of the Brownian movement of molecules. He investigated the thermal properties of light with a low radiation density and his observations laid the foundation of the photon theory of light.\n
In his early days in Berlin, Einstein postulated that the correct interpretation of the special theory of relativity must also furnish a theory of gravitation and in 1916 he published his paper on the general theory of relativity. During this time he also contributed to the problems of the theory of radiation and statistical mechanics.\n
In the 1920s, Einstein embarked on the construction of unified field theories, although he continued to work on the probabilistic interpretation of quantum theory, and he persevered with this work in America. He contributed to statistical mechanics by his development of the quantum theory of a monatomic gas and he has also accomplished valuable work in connection with atomic transition probabilities and relativistic cosmology.\n
After his retirement he continued to work towards the unification of the basic concepts of physics, taking the opposite approach, geometrisation, to the majority of physicists. Einstein’s researches are, of course, well chronicled and his more important works include Special Theory of Relativity (1905), Relativity (English translations, 1920 and 1950), General Theory of Relativity (1916), Investigations on Theory of Brownian Movement (1926), and The Evolution of Physics (1938).\n
Among his non-scientific works, About Zionism (1930), Why War? (1933), My Philosophy (1934), and Out of My Later Years (1950) are perhaps the most important.\n
Albert Einstein received honorary doctorate degrees in science, medicine and philosophy from many European and American universities. During the 1920’s he lectured in Europe, America and the Far East, and he was awarded Fellowships or Memberships of all the leading scientific academies throughout the world. He gained numerous awards in recognition of his work, including the Copley Medal of the Royal Society of London in 1925, and the Franklin Medal of the Franklin Institute in 1935.\n
Einstein’s gifts inevitably resulted in his dwelling much in intellectual solitude and, for relaxation, music played an important part in his life.\n
He married Mileva Maric in 1903 and they had a daughter and two sons; their marriage was dissolved in 1919 and in the same year he married his cousin, Elsa Löwenthal, who died in 1936. He died on April 18, 1955 at Princeton, New Jersey.
"""

In [3]:
chars = list((set(text)))
vocab_size = len(chars)

In [4]:
print("Total Characters :", len(text))
print("Vocabulary Size  :", vocab_size)

Total Characters : 4780
Vocabulary Size  : 73


In [5]:
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

In [6]:
hidden_size = 128
seq_length = 50
learning_rate = 0.003
iterations = 20000

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device :", device)

Using Device : cpu


In [7]:
class VanillaRNN(nn.Module):

    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.i2h = nn.Linear(
            vocab_size + hidden_size,
            hidden_size
        )                                       # Input -> Hidden               
        self.h2o = nn.Linear(
            hidden_size,
            vocab_size
        )                                      # Hidden -> Output

    def forward(self, x, hidden):
        combined = torch.cat((x, hidden), dim=1)# Concatenate input + previous hidden state

        # Hidden state
        hidden = torch.tanh(
            self.i2h(combined)
        )

        # Output logits
        output = self.h2o(hidden)

        return output, hidden

    def init_hidden(self):

        return torch.zeros(
            1,
            self.hidden_size
        ).to(device)

In [8]:
model = VanillaRNN(
    vocab_size,
    hidden_size,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

criterion = nn.CrossEntropyLoss()

In [9]:
# ONE HOT ENCODING

def char_tensor(char):
    tensor = torch.zeros(
        1,
        vocab_size
    ).to(device)
    tensor[0][char_to_ix[char]] = 1

    return tensor


In [11]:
def train(inputs, targets, hidden):

    optimizer.zero_grad()

    loss = 0

    # Process sequence step-by-step
    for i in range(len(inputs)):

        x = char_tensor(inputs[i])

        target = torch.tensor(
            [char_to_ix[targets[i]]]
        ).to(device)

        # Forward pass
        output, hidden = model(x, hidden)

        # Cross entropy loss
        loss += criterion(output, target)

    # =====================================================
    # BACKPROPAGATION THROUGH TIME (BPTT)
    # =====================================================

    loss.backward()

    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=5
    )

    optimizer.step()

    return loss.item() / len(inputs), hidden.detach()

In [12]:
def sample(start_char='A', length=200):

    model.eval()

    hidden = model.init_hidden()

    current_char = start_char

    generated_text = current_char

    with torch.no_grad():

        for _ in range(length):

            x = char_tensor(current_char)

            output, hidden = model(x, hidden)

            # Convert logits -> probabilities
            probs = F.softmax(
                output,
                dim=1
            )

            # Sample next character
            idx = torch.multinomial(
                probs,
                num_samples=1
            ).item()

            current_char = ix_to_char[idx]

            generated_text += current_char

    model.train()

    return generated_text

In [13]:
print("\nTraining Started...\n")

pointer = 0

for iteration in range(iterations):

    # Reset at end of dataset
    if pointer + seq_length + 1 >= len(text):

        pointer = 0

    # Input sequence
    input_seq = text[
        pointer:pointer + seq_length
    ]

    # Target sequence (shifted by 1)
    target_seq = text[
        pointer + 1:pointer + seq_length + 1
    ]

    # Initial hidden state
    hidden = model.init_hidden()

    # Train step
    loss, hidden = train(
        input_seq,
        target_seq,
        hidden
    )

    # Move through dataset
    pointer += seq_length

    # =====================================================
    # PRINT PROGRESS
    # =====================================================

    if iteration % 1000 == 0:

        print("=" * 60)
        print(f"Iteration : {iteration}")
        print(f"Loss      : {loss:.4f}")

        print("\nGenerated Text:\n")

        print(
            sample(
                start_char='A',
                length=300
            )
        )

        print("\n")

print("Training Finished.")


Training Started...

Iteration : 0
Loss      : 4.2846

Generated Text:

ACkWC-8wo1UxyYDr*HHG)1U6qd);3n)4FPYk7OxügBUB(ZfHL::’jü)*(*Ht*flp*6-W,rUwoqehR’agclD8UmGqDf’PByb-CZtE’2OLDkAscW9hK4d-urqbcHrenpRpSlpcgSE8y2OlZc:6ueiWIKrf7P7YtNO3EMGtLcp8FpvHdLvq6r40f6 4hsbKd6ganMiIEw0aMN4wsGdwörMz-vb6lZ.4x)x5)gGImdLW3ywz9ax:1?Jü3PKNPa
MsbTto*cw(5xfr;DTC,pK-wö2öLxd6eEmo:aRtcwhR335c.pJC


Iteration : 1000
Loss      : 1.7066

Generated Text:

Aoment of9quetted thered why mcon the an ppor pithist the fthe wathisl the mpaltelsed ind his aed sf wonv meastof an 

ce ted tiems reollon thiontils ofechita tu isprutis pponporle., ind ts and the prowad ojowat of bhis tut mpmomecl te has poniclotical m phpollt on the wow trochiticsteals ad proberim


Iteration : 2000
Loss      : 1.2820

Generated Text:

Albar  ppo(Alen.

Amant ina n ais d the qhyork Whys mOpr l19In Intymany y. He reale pare be ane mperine.

He shevis divel he d In Eim it Zun 19190h h of mecriean, pe 180l OFanila works indler whis dectuep hious 

# LSTM

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# =========================================================
# 1. DATA (Exactly the same)
# =========================================================

# (same data )

chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}


In [15]:
# =========================================================
# 2. HYPERPARAMETERS
# =========================================================
hidden_size = 128
seq_length = 50
learning_rate = 0.1
iterations = 20000

# =========================================================
# 3. THE PYTORCH MODEL
# =========================================================
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(CharLSTM, self).__init__()
        self.hidden_size = hidden_size
        
        # PyTorch has the entire LSTM math (Forget, Input, Output gates) built in!
        # batch_first=True means we format our data as (batch, sequence, features)
        self.lstm = nn.LSTM(input_size=vocab_size, hidden_size=hidden_size, batch_first=True)
        
        # The Hidden -> Output layer
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden_states):
        # Pass data through the LSTM conveyor belt
        out, (h, c) = self.lstm(x, hidden_states)
        
        # Reshape the output to pass it through the Linear layer
        out = out.reshape(-1, self.hidden_size)
        out = self.fc(out)
        
        return out, (h, c)

# Initialize our model
model = CharLSTM(vocab_size, hidden_size)

In [16]:
# =========================================================
# 4. OPTIMIZER & LOSS FUNCTION
# =========================================================
# PyTorch combines Softmax and CrossEntropy into one step for better stability
criterion = nn.CrossEntropyLoss()

# PyTorch handles the Adagrad memory and math internally
optimizer = optim.Adagrad(model.parameters(), lr=learning_rate)

# =========================================================
# 5. TEXT SAMPLING
# =========================================================
def sample(model, h, c, seed_ix, n):
    model.eval() # Tell PyTorch we are evaluating, not training
    with torch.no_grad(): # Turn off Autograd to save memory
        
        # Shape: (batch=1, seq_len=1, vocab_size)
        x = torch.zeros(1, 1, vocab_size)
        x[0, 0, seed_ix] = 1.0
        hidden = (h, c)
        generated_chars = []

        for _ in range(n):
            out, hidden = model(x, hidden)
            
            # Apply Softmax to get probabilities
            p = torch.nn.functional.softmax(out, dim=1).numpy().ravel()
            ix = np.random.choice(range(vocab_size), p=p)
            
            # Prepare next input
            x = torch.zeros(1, 1, vocab_size)
            x[0, 0, ix] = 1.0
            generated_chars.append(ix_to_char[ix])
            
    model.train() # Switch back to training mode
    return ''.join(generated_chars)

# =========================================================
# 6. TRAINING LOOP
# =========================================================
n, p = 0, 0

# PyTorch expects hidden states in the shape: (num_layers, batch_size, hidden_size)
hprev = torch.zeros(1, 1, hidden_size)
Cprev = torch.zeros(1, 1, hidden_size)

print("\nPyTorch LSTM Training Started...\n")

while n < iterations:
    # Reset at end of data
    if p + seq_length + 1 >= len(text) or n == 0:
        hprev = torch.zeros(1, 1, hidden_size)
        Cprev = torch.zeros(1, 1, hidden_size)
        p = 0

    # Get data chunks
    inputs = [char_to_ix[ch] for ch in text[p:p + seq_length]]
    targets = [char_to_ix[ch] for ch in text[p + 1:p + seq_length + 1]]

    # One-hot encode inputs into a Tensor
    inputs_tensor = torch.zeros(1, len(inputs), vocab_size)
    for t, ix in enumerate(inputs):
        inputs_tensor[0, t, ix] = 1.0
        
    targets_tensor = torch.tensor(targets, dtype=torch.long)

    # --- THE PYTORCH MAGIC HAPPENS HERE ---
    
    # 1. Detach hidden states from history (crucial step!)
    hprev = hprev.detach()
    Cprev = Cprev.detach()

    # 2. Clear old gradients
    optimizer.zero_grad()

    # 3. Forward pass
    outputs, (hprev, Cprev) = model(inputs_tensor, (hprev, Cprev))

    # 4. Calculate Loss
    loss = criterion(outputs, targets_tensor)

    # 5. Backward pass (BPTT - calculus done automatically!)
    loss.backward()

    # 6. Gradient Clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

    # 7. Update weights
    optimizer.step()
    
    # --------------------------------------

    p += seq_length

    if n % 1000 == 0:
        print("=" * 60)
        print(f"Iteration : {n} | Loss: {loss.item():.4f}")
        sample_text = sample(model, hprev, Cprev, inputs[0], 200)
        print(f"\nGenerated Text:\n{sample_text}\n")
        
    n += 1

print("\nTraining Finished.")


PyTorch LSTM Training Started...

Iteration : 0 | Loss: 4.2917

Generated Text:
Ms  übt bnt s nWn Gbt e tor    tibs ttminn,,ne, en ln loii tnnt,tl,t atttt sl  E,tininb ts eln sm btm,nl wn tte  o,  bnt atnm  ,tlrmilntnt enm    t bts    t semE ,irl  nn ü  b brtgW iisb  sb eWs n bem

Iteration : 1000 | Loss: 1.1858

Generated Text:
astingepty ad hise peor and his reexationtand the Browning essting problemsmofn ther wh he ofyedstow anl alsody ow Neatin to relited and massiens an of hish alde.

Einstein wass of the eslppretawis an

Iteration : 2000 | Loss: 0.6136

Generated Text:
ofndev taf ssted Swasude Ulived the Prasid, and his profisailas degmezenstof colishing Einstein wish the reading blst im the Loberthon murerer frormand in 1879 0as. He 6ibeld cinstic post.

He sciste.

Iteration : 3000 | Loss: 0.2322

Generated Text:
inn hiss unstragy his lad acsorblemt im the Prest.

In 1905,.

EInstein’s post in 1915 Pr the forldy Eunstein latizen Spasis poctory of relatidiin muchvend and Inrhis

# Benchmarking Script

### <span style="color:#0275D8">Performance Benchmark: NumPy vs. PyTorch Deep Dive</span>

When transitioning from from-scratch NumPy code to PyTorch's production-ready modules, you face a massive shift in performance dynamics. This comes down to two major axes: **Computation Speed** and **Memory Footprint**.

---

## <span style="color:#5CB85C">1. Execution Time (The PyTorch Advantage)</span>

For sequence modeling workloads, **PyTorch is substantially faster**, often outperforming raw NumPy by **2x to 5x on a standard CPU**.

* <span style="color:#D9534F">**The NumPy Bottleneck:**</span> In raw NumPy, both forward and backward passes rely on standard Python loops: 
  `for t in range(len(inputs)):`
  Python loops are notoriously slow because the interpreter must process them sequentially, handling dynamic typing checking and the **Global Interpreter Lock (GIL)** at every single time step.
* <span style="color:#5CB85C">**The PyTorch Solution:**</span> PyTorch drops execution directly down to a highly optimized C++ backend. When `nn.RNN` executes the classic formula:

$$h_t = \tanh(W_{ih}x_t + b_{ih} + W_{hh}h_{t-1} + b_{hh})$$

Instead of iterating through steps in Python, PyTorch passes the entire input tensor batch to its underlying C++ engine (**ATen**). ATen unrolls the sequence internally, utilizing vectorized CPU instructions (like AVX/SIMD) to execute parallelized matrix operations.

---

## <span style="color:#F0AD4E"> 2. Memory Allocation (The PyTorch Tax)</span>

Conversely, **PyTorch consumes significantly more memory**, often requiring **10x to 50x more peak RAM** than NumPy for small-scale models.

* <span style="color:#5BC0DE">**NumPy Efficiency:**</span> In NumPy, you have explicit control. You overwrite old variables, drop scopes, or clear cache dictionaries manually, keeping the active memory footprint incredibly tiny.
* <span style="color:#D9534F">**The PyTorch Autograd Tax:**</span> To automate gradient computation, PyTorch utilizes **Autograd**. During every single forward pass, PyTorch implicitly constructs a dynamic **Directed Acyclic Graph (DAG)** in memory. 

<div class="alert alert-block alert-warning" style="border-left: 5px solid #F0AD4E;">
    <b>The Tradeoff:</b> The engine caches the exact inputs, intermediate states, and outputs of every single mathematical operation across the entire sequence sequence unrolling. This massive cache is required so PyTorch can safely traverse the graph backwards when you trigger <code>loss.backward()</code>. You are essentially paying a steep RAM tax to ensure you never have to derive complex multi-variable calculus matrices by hand again.
</div>


## 3. code

In [17]:
import numpy as np
import torch
import torch.nn as nn
import time
import tracemalloc

# ==========================================
# 1. HYPERPARAMETERS & DUMMY DATA
# ==========================================
vocab_size = 65
hidden_size = 128
seq_length = 50
iterations = 1000

# Create dummy sequence data (integers representing characters)
np_inputs = np.random.randint(0, vocab_size, size=(seq_length,))
np_targets = np.random.randint(0, vocab_size, size=(seq_length,))

# PyTorch requires specific tensor shapes: (batch_size, seq_length, input_size)
# For nn.CrossEntropyLoss, targets are usually (batch_size, seq_length)
pt_inputs = torch.nn.functional.one_hot(torch.tensor(np_inputs), num_classes=vocab_size).float().unsqueeze(0)
pt_targets = torch.tensor(np_targets).unsqueeze(0)

# ==========================================
# 2. NUMPY RNN (Raw Math)
# ==========================================
# Initialize weights
Wxh = np.random.randn(hidden_size, vocab_size) * 0.01
Whh = np.random.randn(hidden_size, hidden_size) * 0.01
Why = np.random.randn(vocab_size, hidden_size) * 0.01
bh = np.zeros((hidden_size, 1))
by = np.zeros((vocab_size, 1))

def numpy_bptt(inputs, targets, hprev):
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(hprev)
    loss = 0
    
    # Forward Pass
    for t in range(len(inputs)):
        xs[t] = np.zeros((vocab_size, 1))
        xs[t][inputs[t]] = 1
        hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh)
        ys[t] = np.dot(Why, hs[t]) + by
        
        # Softmax
        exp_y = np.exp(ys[t] - np.max(ys[t]))
        ps[t] = exp_y / np.sum(exp_y)
        loss += -np.log(ps[t][targets[t], 0] + 1e-8)
        
    # Backward Pass
    dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)
    dhnext = np.zeros_like(hs[0])
    
    for t in reversed(range(len(inputs))):
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1
        
        dWhy += np.dot(dy, hs[t].T)
        dby += dy
        dh = np.dot(Why.T, dy) + dhnext
        dhraw = (1 - hs[t] * hs[t]) * dh 
        dbh += dhraw
        dWxh += np.dot(dhraw, xs[t].T)
        dWhh += np.dot(dhraw, hs[t-1].T)
        dhnext = np.dot(Whh.T, dhraw)
        
    return loss, dWxh, dWhh, dWhy, dbh, dby

# ==========================================
# 3. PYTORCH RNN (Framework)
# ==========================================
class PyTorchRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(PyTorchRNN, self).__init__()
        self.rnn = nn.RNN(input_size=vocab_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        out, h = self.rnn(x)
        # Reshape for CrossEntropyLoss
        out = self.fc(out)
        # Permute to match (batch_size, num_classes, seq_length) for Loss function
        return out.permute(0, 2, 1)

pt_model = PyTorchRNN(vocab_size, hidden_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(pt_model.parameters(), lr=0.1)

# ==========================================
# 4. THE BENCHMARK
# ==========================================
print(f"--- Running Benchmark: {iterations} Iterations ---\n")

# --- NUMPY TEST ---
tracemalloc.start()
start_time = time.perf_counter()

hprev = np.zeros((hidden_size, 1))
for _ in range(iterations):
    loss, dWxh, dWhh, dWhy, dbh, dby = numpy_bptt(np_inputs, np_targets, hprev)
    # Simulate a basic weight update to make it fair
    Wxh -= 0.1 * dWxh

np_time = time.perf_counter() - start_time
_, np_peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("NUMPY (Raw Python):")
print(f"Execution Time : {np_time:.4f} seconds")
print(f"Peak Memory    : {np_peak_mem / 1024:.2f} KB\n")

# --- PYTORCH TEST ---
tracemalloc.start()
start_time = time.perf_counter()

for _ in range(iterations):
    optimizer.zero_grad()
    outputs = pt_model(pt_inputs)
    loss = criterion(outputs, pt_targets)
    loss.backward()
    optimizer.step()

pt_time = time.perf_counter() - start_time
_, pt_peak_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("PYTORCH (Framework):")
print(f"Execution Time : {pt_time:.4f} seconds")
print(f"Peak Memory    : {pt_peak_mem / 1024:.2f} KB\n")

# --- RESULTS ---
print("--- SUMMARY ---")
if np_time > pt_time:
    print(f"Speed : PyTorch is {np_time / pt_time:.2f}x faster.")
else:
    print(f"Speed : NumPy is {pt_time / np_time:.2f}x faster.")
    
print(f"Memory: PyTorch used {pt_peak_mem / np_peak_mem:.2f}x more memory.")

--- Running Benchmark: 1000 Iterations ---

NUMPY (Raw Python):
Execution Time : 16.3618 seconds
Peak Memory    : 820.17 KB

PYTORCH (Framework):
Execution Time : 7.0979 seconds
Peak Memory    : 28.55 KB

--- SUMMARY ---
Speed : PyTorch is 2.31x faster.
Memory: PyTorch used 0.03x more memory.
